# 04 Permissions and Grants

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Control who can do what in Unity Catalog: the main privileges (<code>USE CATALOG</code>, <code>USE SCHEMA</code>, <code>SELECT</code>, <code>MODIFY</code>, <code>CREATE TABLE</code>, <code>READ VOLUME</code>, <code>EXECUTE</code>, <code>ALL PRIVILEGES</code>, <code>MANAGE</code>), <code>GRANT</code> / <code>REVOKE</code> / <code>SHOW GRANTS</code>, ownership, and how privileges are <b>inherited</b> down the hierarchy. You'll also model access in Python to reason about "can this user read this table?".
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Access control is the core of governance and a large part of the exam's 15% governance domain. In real teams, "I can see the table but get PERMISSION_DENIED" is a daily question, and the answer is almost always a missing privilege on a parent object.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How Unity Catalog permissions work</b>
<ul><li><b>Principals</b>: users, service principals and groups (grant to <b>groups</b>, not individuals)</li><li><b>Securables</b>: metastore, catalog, schema, table, view, volume, function, external location, ...</li><li><b>Inheritance</b>: a privilege granted on a catalog or schema applies to all current <b>and future</b> objects inside it</li><li><b>Access requires the whole path</b>: to read <code>c.s.t</code> you need <code>USE CATALOG</code> on <code>c</code>, <code>USE SCHEMA</code> on <code>c.s</code>, and <code>SELECT</code> on <code>t</code> (or on <code>s</code> or <code>c</code>, inherited)</li><li><b>Owners</b> have all privileges on their object and can grant them. Metastore admins and catalog owners can manage everything below</li><li>Deny by default: no grant, no access</li></ul>
| Privilege | On | Allows |<br>|---|---|---|<br>| <code>USE CATALOG</code> / <code>USE SCHEMA</code> | catalog / schema | Traversing into it (required, grants no data access by itself) |<br>| <code>SELECT</code> | table, view (or inherited from schema/catalog) | Reading |<br>| <code>MODIFY</code> | table | Insert, update, delete, merge |<br>| <code>CREATE TABLE</code> / <code>CREATE VOLUME</code> / <code>CREATE FUNCTION</code> | schema | Creating objects |<br>| <code>CREATE SCHEMA</code> | catalog | Creating schemas |<br>| <code>READ VOLUME</code> / <code>WRITE VOLUME</code> | volume | Reading or writing files |<br>| <code>EXECUTE</code> | function | Calling a UDF |<br>| <code>BROWSE</code> | catalog | Seeing metadata (in Catalog Explorer, lineage) without data access |<br>| <code>MANAGE</code> | any | Managing grants, renaming, dropping, without being the owner |<br>| <code>ALL PRIVILEGES</code> | any | Every applicable privilege (except <code>MANAGE</code> and ownership) |
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
An analytics team had read access granted table by table to 40 people. Every new Gold table meant a ticket and a delay. The platform team switched to groups and inheritance: <code>GRANT USE CATALOG ON CATALOG prod TO analysts</code>, <code>GRANT USE SCHEMA, SELECT ON SCHEMA prod.gold TO analysts</code>. New Gold tables became readable immediately, joiners got access by being added to the group, and an audit of "who can read Gold?" became one query.
</div>

## Syntax

```sql
GRANT USE CATALOG ON CATALOG prod TO `analysts`;
GRANT USE SCHEMA, SELECT ON SCHEMA prod.gold TO `analysts`;        -- all current and future tables
GRANT SELECT ON TABLE prod.gold.revenue TO `finance`;
GRANT MODIFY ON TABLE prod.silver.orders TO `etl-service-principal`;
REVOKE SELECT ON TABLE prod.gold.revenue FROM `finance`;
SHOW GRANTS ON SCHEMA prod.gold;
SHOW GRANTS `analysts` ON SCHEMA prod.gold;
ALTER TABLE prod.gold.revenue OWNER TO `data-platform-team`;
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses the <code>governance</code> schema and creates a <code>revenue</code> table to grant on.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.governance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
spark.sql(f"CREATE OR REPLACE TABLE {DB}.revenue (region STRING, month STRING, revenue DOUBLE)")
spark.sql(f"INSERT INTO {DB}.revenue VALUES ('IN', '2024-11', 1200.0), ('UK', '2024-11', 950.0), ('AE', '2024-11', 780.0)")

## 1. Who am I, and who owns this?

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows your user name and the owner of the <code>revenue</code> table.<br><br>
<b>Why it matters</b><br>The creator of an object becomes its owner and has all privileges on it. That's why you can do everything with your own tables in Free Edition.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Your email address, and the same address as the table owner.
</div>

In [0]:
me = spark.sql("SELECT current_user()").first()[0]
print("current user:", me)
owner = spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.revenue").filter("col_name = 'Owner'").first()
print("table owner: ", owner["data_type"] if owner else "(not shown)")

## 2. Grant, show and revoke

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Grants <code>SELECT</code> on the table to the built-in <code>account users</code> group (everyone in the account), shows the grants, then revokes it.<br><br>
<b>Why it matters</b><br><code>GRANT</code> / <code>SHOW GRANTS</code> / <code>REVOKE</code> are the everyday commands. Granting to a group means membership changes don't require new grants.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
In Free Edition you're the only user, but grants still work and <code>SHOW GRANTS</code> reflects them. Group management (creating groups, adding members) is an account-level task that Free Edition may not expose, so the built-in <code>account users</code> group is used.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
After <code>GRANT</code>, <code>SHOW GRANTS</code> includes a <code>SELECT</code> row for <code>account users</code>. After <code>REVOKE</code>, that row is gone.
</div>

In [0]:
def grants(securable):
    try:
        return [(r["Principal"], r["ActionType"]) for r in spark.sql(f"SHOW GRANTS ON {securable}").collect()]
    except Exception as e:
        return f"SHOW GRANTS not available: {type(e).__name__}"

try:
    spark.sql(f"GRANT SELECT ON TABLE {DB}.revenue TO `account users`")
    print("after GRANT: ", grants(f"TABLE {DB}.revenue"))
    spark.sql(f"REVOKE SELECT ON TABLE {DB}.revenue FROM `account users`")
    print("after REVOKE:", grants(f"TABLE {DB}.revenue"))
except Exception as e:
    print("GRANT not permitted here:", type(e).__name__, str(e)[:200])

## 3. Inheritance: grant once on the schema

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Grants <code>USE SCHEMA</code> and <code>SELECT</code> on the whole schema, then creates a <b>new</b> table and checks that the grant applies to it without any extra command.<br><br>
<b>Why it matters</b><br>Schema- and catalog-level grants cover future objects, so new tables don't need new tickets. That's the main tool for keeping permissions simple.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The schema grants are listed. <code>SHOW GRANTS</code> on the new table shows the inherited <code>SELECT</code> (inherited grants are reported with the parent object they come from).
</div>

In [0]:
try:
    spark.sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {DB} TO `account users`")
    print("schema grants:", grants(f"SCHEMA {DB}"))
    spark.sql(f"CREATE OR REPLACE TABLE {DB}.new_gold_table AS SELECT 1 AS x")
    display(spark.sql(f"SHOW GRANTS ON TABLE {DB}.new_gold_table"))
    spark.sql(f"REVOKE USE SCHEMA, SELECT ON SCHEMA {DB} FROM `account users`")
    spark.sql(f"DROP TABLE {DB}.new_gold_table")
except Exception as e:
    print("Schema grants not permitted here:", type(e).__name__, str(e)[:200])

## 4. Model access in Python

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a small access checker that follows the Unity Catalog rules: a principal can read <code>c.s.t</code> only with <code>USE CATALOG</code>, <code>USE SCHEMA</code> and <code>SELECT</code> (directly or inherited), or as an owner. Then answers a few "can X read Y?" questions.<br><br>
<b>Why it matters</b><br>Reasoning about effective access is what the exam and real support tickets ask for. Encoding the rules once makes them concrete: most denials come from a missing <code>USE</code> privilege on a parent.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>analysts</code> can read <code>prod.gold.revenue</code> (schema-level SELECT is inherited), <code>finance</code> can't (no <code>USE SCHEMA</code>), and the owner can.
</div>

In [0]:
GRANTS = {
    ("analysts", "catalog", "prod"): {"USE CATALOG"},
    ("analysts", "schema", "prod.gold"): {"USE SCHEMA", "SELECT"},
    ("finance", "catalog", "prod"): {"USE CATALOG"},
    ("finance", "table", "prod.gold.revenue"): {"SELECT"},          # SELECT, but no USE SCHEMA
}
OWNERS = {"prod.gold.revenue": "data-platform-team"}

def has(principal, level, name, privilege):
    return privilege in GRANTS.get((principal, level, name), set()) or "ALL PRIVILEGES" in GRANTS.get((principal, level, name), set())

def can_select(principal, table):
    catalog, schema, _ = table.split(".")
    if OWNERS.get(table) == principal:
        return True, "owner"
    missing = []
    if not has(principal, "catalog", catalog, "USE CATALOG"):
        missing.append(f"USE CATALOG on {catalog}")
    if not has(principal, "schema", f"{catalog}.{schema}", "USE SCHEMA"):
        missing.append(f"USE SCHEMA on {catalog}.{schema}")
    if not any(has(principal, lvl, n, "SELECT") for lvl, n in [("catalog", catalog), ("schema", f"{catalog}.{schema}"), ("table", table)]):
        missing.append(f"SELECT on {table} (or its schema/catalog)")
    return (not missing), ("ok" if not missing else "missing " + ", ".join(missing))

for who in ["analysts", "finance", "data-platform-team", "interns"]:
    ok, why = can_select(who, "prod.gold.revenue")
    print(f"{who:<20} can read prod.gold.revenue: {ok!s:<5} ({why})")

## 5. Ownership

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows how ownership is transferred, and why tables should be owned by groups rather than individuals.<br><br>
<b>Why it matters</b><br>If a person who owns production tables leaves, their objects become hard to manage. Transferring ownership to a group or service principal avoids that.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The DDL is printed. (Transferring ownership away from yourself in Free Edition would lock you out of managing the table, so it isn't run.)
</div>

In [0]:
print(f"""
ALTER TABLE {DB}.revenue OWNER TO `data-platform-team`;       -- table
ALTER SCHEMA {DB} OWNER TO `data-platform-team`;              -- schema
-- or grant MANAGE so a group can administer grants without owning the object:
GRANT MANAGE ON TABLE {DB}.revenue TO `data-platform-team`;
""")

## 6. Audit who has access

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Queries <code>information_schema</code> for all privileges granted in the <code>governance</code> schema.<br><br>
<b>Why it matters</b><br>"Who can read this?" should be answerable with a query, for audits and access reviews.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Rows of grantee, privilege and object (possibly empty if all grants were revoked above).
</div>

In [0]:
try:
    display(spark.sql(f"""
        SELECT grantee, privilege_type, table_name, inherited_from
        FROM {CATALOG}.information_schema.table_privileges
        WHERE table_schema = 'governance'
        ORDER BY table_name, grantee
    """))
except Exception as e:
    print("table_privileges not available here:", type(e).__name__)

## Under the hood

```
SELECT * FROM prod.gold.revenue   by user u (member of groups G)
  ├─ USE CATALOG on prod            granted to u or any group in G?   no → PERMISSION_DENIED
  ├─ USE SCHEMA on prod.gold        granted to u or any group in G?   no → PERMISSION_DENIED
  ├─ SELECT on table / schema / catalog (inherited)?                  no → PERMISSION_DENIED
  └─ (row filters and column masks applied, lesson 07) → data
owners and MANAGE holders can grant; metastore admins can manage everything
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata</span> grants are stored in Unity Catalog and enforced on every compute that's Unity Catalog-enabled.

In [0]:
spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.revenue").filter("col_name IN ('Catalog', 'Database', 'Table', 'Owner')").show(truncate=False)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>SELECT</code> granted but still <code>PERMISSION_DENIED</code></b><br>
Missing <code>USE CATALOG</code> or <code>USE SCHEMA</code> on the parents. Both are required.<br><br>
<b>⛔ Problem: granting to individual users</b><br>Hard to maintain and audit. Grant to groups, manage membership in groups.<br><br>
<b>⛔ Problem: re-granting on every new table</b><br>Grant at the schema level, so future tables inherit it.<br><br>
<b>⛔ Problem: production tables owned by a person</b><br>Transfer ownership to a group or service principal, or grant <code>MANAGE</code>.<br><br>
**⛔ Problem: <code>ALL PRIVILEGES</code> everywhere "to unblock"**<br>Breaks least privilege. Grant the specific privileges needed.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What privileges does a user need to query a table in Unity Catalog?</b><br>
<code>USE CATALOG</code> on the catalog, <code>USE SCHEMA</code> on the schema, and <code>SELECT</code> on the table, either directly or inherited from the schema or catalog. Owners already have all privileges on their objects.<br><br>

<b>🎤 2. How does privilege inheritance work?</b><br>
A privilege granted on a catalog or schema applies to all current and future objects of the relevant type inside it. For example, <code>SELECT</code> on a schema lets the grantee read every table and view in it, including ones created later.<br><br>

<b>🎤 3. How would you set up read access for an analytics team to Gold tables?</b><br>
Create or use a group, then <code>GRANT USE CATALOG</code> on the catalog and <code>GRANT USE SCHEMA, SELECT</code> on the Gold schema to that group. People get access by group membership, and new Gold tables are covered by inheritance.<br><br>

<b>🎤 4. What is the difference between ownership and <code>MANAGE</code>?</b><br>
The owner has all privileges and can grant them. There is one owner per object. <code>MANAGE</code> lets other principals manage privileges, rename or drop the object without being the owner, which is useful for platform teams.<br><br>

<b>🎤 5. What does <code>USE SCHEMA</code> allow on its own?</b><br>
Only traversing into the schema. It grants no access to data in the tables, so it's always combined with privileges like <code>SELECT</code>.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. An analyst has <code>SELECT</code> on <code>prod.sales.orders</code> but gets <code>PERMISSION_DENIED</code>. What is most likely missing?</b><br>
A. <code>MODIFY</code> on the table<br>
B. <code>USE CATALOG</code> on <code>prod</code> and/or <code>USE SCHEMA</code> on <code>prod.sales</code><br>
C. <code>CREATE TABLE</code> on the schema<br>
D. Ownership of the table<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which command lets the <code>analysts</code> group read all current and future tables in <code>prod.gold</code>?</b><br>
A. <code>GRANT SELECT ON TABLE prod.gold.* TO analysts</code><br>
B. <code>GRANT USE SCHEMA, SELECT ON SCHEMA prod.gold TO analysts</code> (plus <code>USE CATALOG</code> on <code>prod</code>)<br>
C. <code>GRANT ALL PRIVILEGES ON METASTORE TO analysts</code><br>
D. <code>ALTER SCHEMA prod.gold OWNER TO analysts</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q3. Which statement removes a previously granted privilege?</b><br>
A. <code>DENY</code><br>
B. <code>REVOKE</code><br>
C. <code>DROP GRANT</code><br>
D. <code>UNGRANT</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add a principal <code>auditors</code> to the Python model with <code>USE CATALOG</code> on <code>prod</code> and <code>SELECT</code> on the whole <code>prod</code> catalog. Can they read <code>prod.gold.revenue</code>? Why not, and what single grant fixes it?</li><li>Add <code>BROWSE</code>-only access for <code>interns</code> and check that <code>can_select</code> still says no</li><li>(Databricks) Grant <code>SELECT</code> on <code>revenue</code> to <code>account users</code>, list grants with <code>information_schema.table_privileges</code>, and revoke it</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Catalog-level SELECT is inherited, but USE SCHEMA is still required
GRANTS[("auditors", "catalog", "prod")] = {"USE CATALOG", "SELECT"}
print("auditors before:", can_select("auditors", "prod.gold.revenue"))
GRANTS[("auditors", "schema", "prod.gold")] = {"USE SCHEMA"}
print("auditors after USE SCHEMA:", can_select("auditors", "prod.gold.revenue"))

# 2. BROWSE gives metadata visibility only
GRANTS[("interns", "catalog", "prod")] = {"BROWSE"}
print("interns:", can_select("interns", "prod.gold.revenue"))

In [0]:
# 3. Grant, audit, revoke
try:
    spark.sql(f"GRANT SELECT ON TABLE {DB}.revenue TO `account users`")
    display(spark.sql(f"SELECT grantee, privilege_type FROM {CATALOG}.information_schema.table_privileges WHERE table_schema = 'governance' AND table_name = 'revenue'"))
    spark.sql(f"REVOKE SELECT ON TABLE {DB}.revenue FROM `account users`")
except Exception as e:
    print("Not permitted here:", type(e).__name__)

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Reading <code>c.s.t</code> needs <code>USE CATALOG</code> + <code>USE SCHEMA</code> + <code>SELECT</code> (direct or inherited)</li><li>Grants on catalogs and schemas are inherited by current and future objects</li><li>Grant to groups. Keep ownership with groups or service principals, or grant <code>MANAGE</code></li><li><code>GRANT</code>, <code>REVOKE</code>, <code>SHOW GRANTS</code>, <code>information_schema.*_privileges</code></li><li>Least privilege: specific privileges, not <code>ALL PRIVILEGES</code></li></ul>
</div>

| Task | Command |
|---|---|
| Read a schema | `GRANT USE SCHEMA, SELECT ON SCHEMA c.s TO g` (+ `USE CATALOG`) |
| Write a table | `GRANT MODIFY ON TABLE c.s.t TO g` |
| Create tables | `GRANT CREATE TABLE ON SCHEMA c.s TO g` |
| Files | `GRANT READ VOLUME / WRITE VOLUME ON VOLUME c.s.v TO g` |
| Remove | `REVOKE ... FROM g` |
| Inspect | `SHOW GRANTS ON TABLE c.s.t` |
| Owner | `ALTER TABLE c.s.t OWNER TO g` |

## Git commit

```
git add 06_unity_catalog/04_permissions_grants.ipynb
git commit -m "add 06_04 permissions and grants notebook"
```